In [1]:
import pandas as pd
import numpy as np
import os
import shutil
from sklearn.model_selection import train_test_split
from pathlib import Path
import glob

def load_hmc_data():
    """HMC 데이터셋 로드"""
    hmc_path = "/home/work/SCOUTER/scouter/data/HMC"
    annotation_file = os.path.join(hmc_path, "MasterlistAug30-2017.xlsx")
    
    # 어노테이션 파일 로드
    df = pd.read_excel(annotation_file)
    
    # Image와 Outcome 컬럼 사용
    hmc_df = df[['Image', 'Outcome']].copy()
    hmc_df = hmc_df[hmc_df['Outcome'].isin([0.0, 1.0])]
    
    # 컬럼명 통일 및 이미지 경로 생성
    hmc_df.columns = ['Image', 'label']
    hmc_df['dataset'] = 'HMC'
    hmc_df['imageID'] = hmc_df['Image'].apply(lambda x: f"/home/work/SCOUTER/scouter/data/HMC/Images/{x}.png" if not x.endswith('.png') else f"/home/work/SCOUTER/scouter/data/HMC/Images/{x}")
    
    return hmc_df

def load_kromp_data():
    """Kromp 데이터셋 로드"""
    kromp_path = "/home/work/SCOUTER/scouter/data/Kromp"
    annotation_file = os.path.join(kromp_path, "Clincial_annotations.csv")
    
    # 어노테이션 파일 로드 (인코딩 문제 해결)
    encodings_to_try = ['utf-8', 'cp949', 'euc-kr', 'latin-1', 'iso-8859-1']
    separators_to_try = [',', ';', '\t']
    df = None
    
    for encoding in encodings_to_try:
        for sep in separators_to_try:
            try:
                df = pd.read_csv(annotation_file, encoding=encoding, sep=sep)
                if len(df.columns) > 5:
                    break
            except:
                continue
        if df is not None and len(df.columns) > 5:
            break
    
    if df is None or len(df.columns) <= 5:
        raise ValueError("지원하는 인코딩과 구분자로 파일을 읽을 수 없습니다.")
    
    # 컬럼명에 공백 제거
    df.columns = df.columns.str.strip()
    
    # d=5인 데이터만 필터링
    if 'd' in df.columns:
        df = df[df['d'] == 5].copy()
    
    # Image와 SS 컬럼만 사용
    kromp_df = df[['Image', 'SS']].copy()
    
    # 컬럼명 통일 및 이미지 경로 생성
    kromp_df.columns = ['Image', 'label']
    kromp_df['dataset'] = 'Kromp'
    kromp_df['imageID'] = kromp_df['Image'].apply(lambda x: f"/home/work/SCOUTER/scouter/data/Kromp/Images/{x}")
    
    return kromp_df

def create_embryo_dataset():
    """Embryo 데이터셋 생성 (HMC + Kromp)"""
    print("=== Embryo 데이터셋 생성 ===")
    
    # 데이터 로드
    hmc_df = load_hmc_data()
    kromp_df = load_kromp_data()
    
    # 데이터 합치기
    embryo_df = pd.concat([hmc_df, kromp_df], ignore_index=True)
    
    print(f"Embryo 데이터셋 구성:")
    print(f"  - HMC 데이터: {len(hmc_df)}장")
    print(f"  - Kromp 데이터: {len(kromp_df)}장")
    print(f"  - 총 데이터: {len(embryo_df)}장")
    
    # 라벨 분포 확인
    print(f"Embryo 라벨 분포:")
    label_dist = embryo_df['label'].value_counts().sort_index()
    for label, count in label_dist.items():
        label_name = 'success' if label == 1.0 else 'failure'
        print(f"  - {label_name} (label={label}): {count}장")
    
    return embryo_df

def create_transfer_dataset():
    """Transfer 데이터셋 생성 (HMC Transfer + Kromp)"""
    print("=== Transfer 데이터셋 생성 ===")
    
    # HMC Transfer 데이터 처리
    hmc_transfer_path = "/home/work/SCOUTER/scouter/data/HMC Transfer"
    hmc_transfer_images = glob.glob(os.path.join(hmc_transfer_path, "*.png"))
    
    # HMC 원본 어노테이션 사용
    hmc_df = load_hmc_data()
    
    # HMC Transfer 이미지 파일명에서 원본 파일명 매칭
    hmc_transfer_df = []
    matched_count = 0
    unmatched_images = []
    
    for img_path in hmc_transfer_images:
        img_name = os.path.basename(img_path)
        
        # 확장자 제거 후 매칭
        img_name_no_ext = os.path.splitext(img_name)[0]
        
        # HMC 어노테이션에서 확장자 제거 후 매칭
        matching_row = None
        for idx, hmc_row in hmc_df.iterrows():
            hmc_name = str(hmc_row['Image'])
            hmc_name_no_ext = os.path.splitext(hmc_name)[0]
            
            if img_name_no_ext == hmc_name_no_ext:
                matching_row = hmc_df.loc[[idx]]
                break
        
        if matching_row is not None and not matching_row.empty:
            hmc_transfer_df.append({
                'Image': img_name,
                'label': matching_row.iloc[0]['label'],
                'dataset': 'HMC_Transfer',
                'imageID': f"/home/work/SCOUTER/scouter/data/HMC Transfer/{img_name}"
            })
            matched_count += 1
        else:
            unmatched_images.append(img_name)
    
    hmc_transfer_df = pd.DataFrame(hmc_transfer_df)
    
    # Kromp 데이터 로드
    kromp_df = load_kromp_data()
    
    # 데이터 합치기
    transfer_df = pd.concat([hmc_transfer_df, kromp_df], ignore_index=True)
    
    print(f"Transfer 데이터셋 구성:")
    print(f"  - HMC Transfer 데이터: {len(hmc_transfer_df)}장")
    print(f"    * HMC Transfer 폴더 총 이미지: {len(hmc_transfer_images)}장")
    print(f"    * 어노테이션 매칭 성공: {matched_count}장")
    if unmatched_images:
        print(f"    * 어노테이션 매칭 실패: {len(unmatched_images)}장")
    print(f"  - Kromp 데이터: {len(kromp_df)}장")
    print(f"  - 총 데이터: {len(transfer_df)}장")
    
    # 라벨 분포 확인
    print(f"Transfer 라벨 분포:")
    label_dist = transfer_df['label'].value_counts().sort_index()
    for label, count in label_dist.items():
        label_name = 'success' if label == 1.0 else 'failure'
        print(f"  - {label_name} (label={label}): {count}장")
    
    return transfer_df, hmc_transfer_df

def split_dataset_safe(df, hmc_transfer_df=None, test_size=0.1, val_size=0.2):
    """
    데이터셋을 train/val/test로 분할
    Transfer 데이터셋의 경우 HMC Transfer 이미지는 테스트셋에서 제외
    """
    if hmc_transfer_df is not None:
        # Transfer 데이터셋의 경우: HMC Transfer 이미지는 테스트셋에서 제외
        hmc_transfer_indices = hmc_transfer_df.index.tolist()
        non_transfer_indices = [i for i in df.index if i not in hmc_transfer_indices]
        
        # 테스트셋은 non-transfer 데이터에서만 선택
        non_transfer_df = df.loc[non_transfer_indices]
        test_indices = non_transfer_df.sample(n=int(len(df) * test_size), random_state=42).index
        
        # 나머지 데이터에서 train/val 분할
        remaining_indices = [i for i in df.index if i not in test_indices]
        remaining_df = df.loc[remaining_indices]
        
        train_indices, val_indices = train_test_split(
            remaining_indices, 
            test_size=val_size/(1-test_size), 
            random_state=42, 
            stratify=remaining_df['label']
        )
        
        print(f"Transfer 데이터셋 분할 (Test Leakage 방지):")
        print(f"  - HMC Transfer 이미지 수: {len(hmc_transfer_indices)}장 (테스트셋 제외)")
        print(f"  - 테스트셋은 Kromp 데이터에서만 선택")
        
    else:
        # Embryo 데이터셋의 경우: 일반적인 분할
        train_val_indices, test_indices = train_test_split(
            df.index, 
            test_size=test_size, 
            random_state=42, 
            stratify=df['label']
        )
        
        train_indices, val_indices = train_test_split(
            train_val_indices, 
            test_size=val_size/(1-test_size), 
            random_state=42, 
            stratify=df.loc[train_val_indices, 'label']
        )
    
    train_df = df.loc[train_indices]
    val_df = df.loc[val_indices] 
    test_df = df.loc[test_indices]
    
    print(f"데이터 분할 결과:")
    print(f"  - Train: {len(train_df)}장 ({len(train_df)/len(df)*100:.1f}%)")
    print(f"  - Val: {len(val_df)}장 ({len(val_df)/len(df)*100:.1f}%)")
    print(f"  - Test: {len(test_df)}장 ({len(test_df)/len(df)*100:.1f}%)")
    
    return train_df, val_df, test_df

def create_folder_structure_and_copy_images(df_dict, dataset_name, source_paths):
    """폴더 구조 생성 및 이미지 복사"""
    base_path = f"/home/work/SCOUTER/scouter/data/{dataset_name}"
    
    # 폴더 구조 생성
    for split in ['train', 'val', 'test']:
        for label in ['success', 'failure']:
            folder_path = os.path.join(base_path, split, label)
            os.makedirs(folder_path, exist_ok=True)
    
    # 이미지 복사
    for split, df in df_dict.items():
        for _, row in df.iterrows():
            # 라벨에 따른 폴더명 결정
            label_folder = 'success' if row['label'] == 1.0 else 'failure'
            
            # imageID에서 소스 경로 추출
            source_img_path = row['imageID']
            
            if os.path.exists(source_img_path):
                # 대상 경로
                dest_path = os.path.join(base_path, split, label_folder, row['Image'])
                
                # 이미지 복사
                shutil.copy2(source_img_path, dest_path)
            else:
                print(f"경고: 이미지를 찾을 수 없음 - {source_img_path}")

def main():
    """메인 실행 함수"""
    
    print("데이터셋 합치기 작업을 시작합니다")
    
    # 1. Embryo 데이터셋 생성
    embryo_df = create_embryo_dataset()
    embryo_train, embryo_val, embryo_test = split_dataset_safe(embryo_df)
    
    # Embryo 데이터프레임 저장 (Image, label, imageID 컬럼만)
    embryo_final = embryo_df[['Image', 'label', 'imageID']].copy()
    embryo_final.to_csv('/home/work/SCOUTER/scouter/data/embryo_dataset.csv', index=False)
    print(f"Embryo 데이터셋 CSV 저장: /home/work/SCOUTER/scouter/data/embryo_dataset.csv")
    
    # Embryo 폴더 구조 생성 및 이미지 복사
    embryo_source_paths = [
        "/home/work/SCOUTER/scouter/data/HMC/Images",
        "/home/work/SCOUTER/scouter/data/Kromp/Images"
    ]
    
    print(f"Embryo 폴더 구조 생성 및 이미지 복사 중...")
    create_folder_structure_and_copy_images(
        {'train': embryo_train, 'val': embryo_val, 'test': embryo_test},
        'embryo',
        embryo_source_paths
    )
    
    # 2. Transfer 데이터셋 생성
    transfer_df, hmc_transfer_df = create_transfer_dataset()
    transfer_train, transfer_val, transfer_test = split_dataset_safe(
        transfer_df, 
        hmc_transfer_df
    )
    
    # Transfer 데이터프레임 저장 (Image, label, imageID 컬럼만)
    transfer_final = transfer_df[['Image', 'label', 'imageID']].copy()
    transfer_final.to_csv('/home/work/SCOUTER/scouter/data/transfer_dataset.csv', index=False)
    print(f"Transfer 데이터셋 CSV 저장: /home/work/SCOUTER/scouter/data/transfer_dataset.csv")
    
    # Transfer 폴더 구조 생성 및 이미지 복사
    transfer_source_paths = [
        "/home/work/SCOUTER/scouter/data/HMC Transfer",
        "/home/work/SCOUTER/scouter/data/Kromp/Images"
    ]
    
    print(f"Transfer 폴더 구조 생성 및 이미지 복사 중...")
    create_folder_structure_and_copy_images(
        {'train': transfer_train, 'val': transfer_val, 'test': transfer_test},
        'transfer',
        transfer_source_paths
    )
    
    print("=" * 80)
    print("데이터셋 생성 완료")
    print("=" * 80)
    
    print(f"최종 요약:")
    print(f"1. Embryo 데이터셋: {len(embryo_df)}장")
    print(f"   - Train: {len(embryo_train)}장, Val: {len(embryo_val)}장, Test: {len(embryo_test)}장")
    print(f"2. Transfer 데이터셋: {len(transfer_df)}장")
    print(f"   - Train: {len(transfer_train)}장, Val: {len(transfer_val)}장, Test: {len(transfer_test)}장")
    
    print(f"생성된 파일 및 폴더:")
    print(f"  - /home/work/SCOUTER/scouter/data/embryo_dataset.csv")
    print(f"  - /home/work/SCOUTER/scouter/data/transfer_dataset.csv")
    print(f"  - /home/work/SCOUTER/scouter/data/embryo/ (폴더 구조 + 이미지)")
    print(f"  - /home/work/SCOUTER/scouter/data/transfer/ (폴더 구조 + 이미지)")

if __name__ == "__main__":
    main()

데이터셋 합치기 작업을 시작합니다
=== Embryo 데이터셋 생성 ===
Embryo 데이터셋 구성:
  - HMC 데이터: 171장
  - Kromp 데이터: 642장
  - 총 데이터: 813장
Embryo 라벨 분포:
  - failure (label=0.0): 452장
  - success (label=1.0): 361장
데이터 분할 결과:
  - Train: 568장 (69.9%)
  - Val: 163장 (20.0%)
  - Test: 82장 (10.1%)
Embryo 데이터셋 CSV 저장: /home/work/SCOUTER/scouter/data/embryo_dataset.csv
Embryo 폴더 구조 생성 및 이미지 복사 중...
경고: 이미지를 찾을 수 없음 - /home/work/SCOUTER/scouter/data/Kromp/Images/187_1.png
=== Transfer 데이터셋 생성 ===
Transfer 데이터셋 구성:
  - HMC Transfer 데이터: 171장
    * HMC Transfer 폴더 총 이미지: 249장
    * 어노테이션 매칭 성공: 171장
    * 어노테이션 매칭 실패: 78장
  - Kromp 데이터: 642장
  - 총 데이터: 813장
Transfer 라벨 분포:
  - failure (label=0.0): 452장
  - success (label=1.0): 361장
Transfer 데이터셋 분할 (Test Leakage 방지):
  - HMC Transfer 이미지 수: 171장 (테스트셋 제외)
  - 테스트셋은 Kromp 데이터에서만 선택
데이터 분할 결과:
  - Train: 569장 (70.0%)
  - Val: 163장 (20.0%)
  - Test: 81장 (10.0%)
Transfer 데이터셋 CSV 저장: /home/work/SCOUTER/scouter/data/transfer_dataset.csv
Transfer 폴더 구조 생성 및 이미지 복사 중...
경고: 이미지를 찾